# Cerebrum Weather Assistant Example

This notebook demonstrates how to use the Cerebrum framework as a library to create a weather assistant that provides clothing recommendations based on weather conditions.

## Setup and Imports

First, let's import the necessary modules from the Cerebrum framework.

In [1]:
import sys
from pathlib import Path
from datetime import datetime

# Add the project root to the Python path
project_root = str(Path().resolve().parents[0])
sys.path.append(project_root)

from src.server.persistence.persistence_model_factory import PersistenceModelFactory
from src.server.models.tasks_manager import TasksManager
from src.server.models.task import Task
from src.server.models.message import Message
from src.server.controllers.curried_proxy import CurriedProxy
from src.server.tools.weather_tool import WeatherTool
from src.server.controllers.task_runner import TaskRunner

/opt/anaconda3/envs/cerebrum/lib/python3.11/site-packages/pydantic/_internal/_config.py:295: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  warnings.warn(DEPRECATION_MESSAGE, DeprecationWarning)
/opt/anaconda3/envs/cerebrum/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:502: UserWarning: <built-in function callable> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(
/opt/anaconda3/envs/cerebrum/lib/python3.11/site-packages/crewai_tools/tools/scrapegraph_scrape_tool/scrapegraph_scrape_tool.py:34: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprec

/opt/anaconda3/envs/cerebrum/lib/python3.11/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "schema" in "ToolMetadata" shadows an attribute in parent "BaseModel"
  warnings.warn(
/Users/mb/projects/genbrain/cerebrum/src/server/models/specialized_state.py:30: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  @validator('state_type')
/opt/anaconda3/envs/cerebrum/lib/python3.11/inspect.py:573: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use `model_fields` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  value = getter(object, key)
/opt/anaconda3/envs/cerebrum/lib/python3.11/


🔧 Initializing default tools...

📦 Registering internal tools...
Registering instance internal_file_tools
  Registered method: internal_file_tools.read_documentation
  Registered method: internal_file_tools.save_implementation
  Registered method: internal_file_tools.save_tests
✅ Registered InternalFileTools with 4 methods

☁️ Registering weather tools...
Registering instance weather
  Registered method: weather.get_forecast
  Registered method: weather.get_weather
✅ Registered WeatherTool with 3 methods

🐍 Registering Python execution tool...
Registering instance PyExec
  Registered method: PyExec.run_code
  Registered method: PyExec.run_script
✅ Registered PyExec with 3 methods
Registering instance internal_credentials_tools
  Registered method: internal_credentials_tools.load_credentials
  Registered method: internal_credentials_tools.validate_credentials
✅ Registered InternalCredentialsTools with 3 methods
Registering instance internal_testing_tools
  Registered method: internal_t

2025-03-29 22:41:04.756 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2025-03-29 22:41:04.757 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`


2025-03-29 22:41:04.757 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2025-03-29 22:41:04.757 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


🆕 Creating new persistence model
🆕 Created InMemoryPersistenceModel: id=5812583952
✅ Stored new persistence model in session state: id=5812583952


/opt/anaconda3/envs/cerebrum/lib/python3.11/site-packages/PyPDF2/__init__.py:21: DeprecationWarning: PyPDF2 is deprecated. Please move to the pypdf library instead.
  warnings.warn(


## Initialize Persistence Model

Create an in-memory persistence model to store our tasks and tools.

In [2]:
# Initialize persistence model
persistence_model = PersistenceModelFactory.persistence_model("inmemory")
print("✅ Initialized in-memory persistence model")

2025-03-29 22:41:04.846 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2025-03-29 22:41:04.847 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


🔄 Using existing persistence model from session state: id=5812583952
📊 Existing persistence model has 0 tasks
✅ Initialized in-memory persistence model


## Create and Register Weather Tools

Set up the weather tools and register them with a curried proxy.

In [3]:
# Create weather tool
weather_tool = WeatherTool()

# Create a curried proxy for the tools
curried_proxy = CurriedProxy()
curried_proxy.add_tool("weather.get_weather", weather_tool.get_weather)
curried_proxy.add_tool("weather.get_forecast", weather_tool.get_forecast)

# Save the curried proxy
curried_proxy = persistence_model.curried_proxies.create(curried_proxy)
curried_proxy_id_str = str(curried_proxy.id)
print(f"✅ Created curried proxy with ID: {curried_proxy_id_str}")

✅ Created curried proxy with ID: 055638a2-0bb9-4225-b96d-a50d69491e1a


## Create Weather Task Manager

Create a task manager for weather-related tasks.

In [4]:
# Create TasksManager for weather
weather_manager = TasksManager(
    name="WeatherManager",
    model="gpt-4o-mini",
    description="Manages tasks for weather reporting and clothing recommendations.",
    messages=[
        Message(
            role="system",
            content="You are a helpful assistant."
        )
    ]
)
weather_manager = persistence_model.tasksmanagers.create(weather_manager)
print(f"✅ Created Weather Manager with ID: {weather_manager.id}")

✅ Created Weather Manager with ID: f25216ba-b56b-49b5-84a0-a4280f5f8e83


## Create Clothing Recommendation Task

Create a task that provides clothing recommendations based on weather conditions.

In [5]:
# Create task for clothing recommendations
clothing_task = Task(
    name="WhatToWear",
    description="Provides clothing recommendations based on weather conditions",
    tasksmanager_id=str(weather_manager.id),
    curried_proxy_id=curried_proxy_id_str,
    use_context=True,
    inherit_manager_context=True,
    messages=[
        Message(
            role="system",
            content="""You are a clothing recommendation assistant specialized in weather-appropriate attire. Your role is to provide helpful, practical clothing suggestions based on weather conditions.

When making recommendations, consider:

1. Temperature ranges:
   - Hot (>85°F/30°C): Lightweight, breathable fabrics
   - Warm (70-85°F/21-30°C): Light layers, short sleeves
   - Mild (55-70°F/13-21°C): Light jackets, long sleeves
   - Cool (40-55°F/4-13°C): Sweaters, light coats
   - Cold (25-40°F/-4-4°C): Heavy coats, layers, hats
   - Very cold (<25°F/-4°C): Thermal layers, heavy insulation

2. Precipitation:
   - Rain: Waterproof jackets, umbrellas, water-resistant footwear
   - Snow: Waterproof boots, insulated coats, gloves, hats
   - Sleet/Ice: Non-slip footwear, waterproof outer layers

3. Wind conditions:
   - Windy: Wind-resistant layers, secured accessories
   - Gusty: Avoid loose clothing, use windbreakers

4. Humidity levels:
   - High humidity: Moisture-wicking fabrics, breathable materials
   - Low humidity: Hydrating fabrics, skin protection

5. UV exposure:
   - Sunny: Hats, sunglasses, UV-protective clothing
   - Overcast: Still consider UV protection

6. Activity level:
   - Active: Moisture-wicking, flexible clothing
   - Casual: Comfortable, appropriate layers
   - Formal: Weather-appropriate professional attire

Always ask for specific weather details if not provided. Tailor recommendations to the user's location, planned activities, and personal preferences. Provide practical, specific suggestions rather than general advice."""
        )
    ]
)
clothing_task = persistence_model.tasks.create(clothing_task)
print(f"✅ Created Clothing Recommendation Task with ID: {clothing_task.id}")

✅ Created Clothing Recommendation Task with ID: fef71737-6863-4d8f-a8a6-97dcb409d76c


## Create Task Runner

Initialize the task runner to execute our tasks.

In [6]:
# Create task runner
task_runner = TaskRunner(persistence_model)
await task_runner.initialize()
print("✅ Created Task Runner")

⚙️ Initializing ToolsOperator
🔍 DEBUG: Initializing OpenAI embedding function with model text-embedding-3-small
✅ DEBUG: OpenAI embedding function initialized successfully
Debug - Using persist directory: persistent_state/chroma_db
🔍 DEBUG: Initializing OpenAI embedding function with model text-embedding-3-small
✅ DEBUG: OpenAI embedding function initialized successfully
✅ Created Task Runner


/opt/anaconda3/envs/cerebrum/lib/python3.11/site-packages/pydantic/_internal/_config.py:295: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  warnings.warn(DEPRECATION_MESSAGE, DeprecationWarning)


## Function to Get Weather and Clothing Recommendations

Create a function that takes a location and returns weather information and clothing recommendations.

In [7]:
async def get_weather_and_clothing_recommendations(location: str):
    """Get weather information and clothing recommendations for a location."""
    # Create a new chat session
    session_id = f"session_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    # First message to get weather information
    weather_query = f"What is the weather in {location}?"
    
    # Create a Message object for the weather query
    weather_message = Message(role="user", content=weather_query)
    
    # Run the task with the weather query
    weather_response_data = await task_runner.run_task(
        tasksmanager_id=str(weather_manager.id),
        task_id=str(clothing_task.id),
        running_extra_params={
            "messages": [weather_message],
            "session_id": session_id
        }
    )
    
    # Extract the response from the result
    weather_response = weather_response_data.get("response", "")
    
    # Second message to get clothing recommendations
    clothing_query = f"What should I wear for this weather in {location}?"
    
    # Create a Message object for the clothing query
    clothing_message = Message(role="user", content=clothing_query)
    
    # Run the task with the clothing query
    clothing_response_data = await task_runner.run_task(
        tasksmanager_id=str(weather_manager.id),
        task_id=str(clothing_task.id),
        running_extra_params={
            "messages": [clothing_message],
            "session_id": session_id
        }
    )
    
    # Extract the response from the result
    clothing_response = clothing_response_data.get("response", "")
    
    return {
        "location": location,
        "weather_info": weather_response,
        "clothing_recommendations": clothing_response
    }

## Try It Out

Let's test our weather assistant with different locations.

In [8]:
import asyncio

# Test with New York
result_ny = await get_weather_and_clothing_recommendations("New York")

print(f"\n🌆 {result_ny['location']} Weather Information:")
print(f"{'=' * 50}")
print(result_ny['weather_info'])
print(f"\n👕 Clothing Recommendations:")
print(f"{'=' * 50}")
print(result_ny['clothing_recommendations'])

2025-03-29 22:41:04,881 - 8368244800 - task_runner.py-task_runner:288 - ERROR:   ❌ CrewAI config not found task.crewai_config_id is None



📖 Reading Task fef71737-6863-4d8f-a8a6-97dcb409d76c:
  Type: <class 'src.server.models.task.Task'>
  Is instance of Task: True

📖 Reading TasksManager f25216ba-b56b-49b5-84a0-a4280f5f8e83:
  Type: <class 'src.server.models.tasks_manager.TasksManager'>
  Is instance of TasksManager: True

📝 Starting output capture for task: fef71737-6863-4d8f-a8a6-97dcb409d76c

🔍 DEBUG: get_contexts called with query='What is the weather in New York?', task_id=fef71737-6863-4d8f-a8a6-97dcb409d76c

📖 Reading Task fef71737-6863-4d8f-a8a6-97dcb409d76c:
  Type: <class 'src.server.models.task.Task'>
  Is instance of Task: True
🔄 DEBUG: Syncing task contexts for task fef71737-6863-4d8f-a8a6-97dcb409d76c

🔄 DEBUG: sync_task_contexts called for task_id=fef71737-6863-4d8f-a8a6-97dcb409d76c

🔄 Syncing task contexts for fef71737-6863-4d8f-a8a6-97dcb409d76c
  Creating new task context
  Checking task attributes:
    has use_context: True
    has inherit_manager_context: True
    use_context: True
    inherit_manag

✅ DEBUG: Generated 1 OpenAI embeddings
🔢 DEBUG: First embedding dimensions: 1536
🔢 DEBUG: First 5 values of first embedding: [-0.04350387305021286, -0.014422602020204067, -0.057645443826913834, 0.04842757061123848, -0.0005687400698661804]
🔢 DEBUG: Generated embedding with dimensions: 1536
🔢 DEBUG: First 5 embedding values: [-0.04350387305021286, -0.014422602020204067, -0.057645443826913834, 0.04842757061123848, -0.0005687400698661804]
🔎 DEBUG: Searching in default collection
🔍 DEBUG: search_in_index called with index_name=default, limit=506, filters=None
🔢 DEBUG: Embedding dimensions: 1536
🔢 DEBUG: First 5 embedding values: [-0.04350387305021286, -0.014422602020204067, -0.057645443826913834, 0.04842757061123848, -0.0005687400698661804]
🔍 DEBUG: _ensure_collection called for collection_name=default
🔍 DEBUG: ChromaDB client: <chromadb.api.client.Client object at 0x15ae42c50>
📋 DEBUG: All ChromaDB collections: ['default']
✅ DEBUG: Collection default already exists in indices
📊 DEBUG: Coll

2025-03-29 22:41:08,220 - 8368244800 - task_runner.py-task_runner:288 - ERROR:   ❌ CrewAI config not found task.crewai_config_id is None



✅ Stopped output capture for task: fef71737-6863-4d8f-a8a6-97dcb409d76c

📖 Reading Task fef71737-6863-4d8f-a8a6-97dcb409d76c:
  Type: <class 'src.server.models.task.Task'>
  Is instance of Task: True

📖 Reading TasksManager f25216ba-b56b-49b5-84a0-a4280f5f8e83:
  Type: <class 'src.server.models.tasks_manager.TasksManager'>
  Is instance of TasksManager: True

🔍 DEBUG: get_contexts called with query='What should I wear for this weather in New York?', task_id=fef71737-6863-4d8f-a8a6-97dcb409d76c

📖 Reading Task fef71737-6863-4d8f-a8a6-97dcb409d76c:
  Type: <class 'src.server.models.task.Task'>
  Is instance of Task: True
🔄 DEBUG: Syncing task contexts for task fef71737-6863-4d8f-a8a6-97dcb409d76c

🔄 DEBUG: sync_task_contexts called for task_id=fef71737-6863-4d8f-a8a6-97dcb409d76c

🔄 Syncing task contexts for fef71737-6863-4d8f-a8a6-97dcb409d76c
  Using existing task context
  Checking task attributes:
    has use_context: True
    has inherit_manager_context: True
    use_context: True


✅ DEBUG: Generated 1 OpenAI embeddings
🔢 DEBUG: First embedding dimensions: 1536
🔢 DEBUG: First 5 values of first embedding: [0.027715200558304787, 0.0007191919139586389, -0.05655304342508316, -0.009197470732033253, -0.006332396995276213]
🔢 DEBUG: Generated embedding with dimensions: 1536
🔢 DEBUG: First 5 embedding values: [0.027715200558304787, 0.0007191919139586389, -0.05655304342508316, -0.009197470732033253, -0.006332396995276213]
🔎 DEBUG: Searching in default collection
🔍 DEBUG: search_in_index called with index_name=default, limit=506, filters=None
🔢 DEBUG: Embedding dimensions: 1536
🔢 DEBUG: First 5 embedding values: [0.027715200558304787, 0.0007191919139586389, -0.05655304342508316, -0.009197470732033253, -0.006332396995276213]
🔍 DEBUG: _ensure_collection called for collection_name=default
🔍 DEBUG: ChromaDB client: <chromadb.api.client.Client object at 0x15ae42c50>
📋 DEBUG: All ChromaDB collections: ['default']
✅ DEBUG: Collection default already exists in indices
📊 DEBUG: Coll


🌆 New York Weather Information:
The current weather in New York is as follows:

- **Temperature**: 26°C
- **Feels Like**: 26°C
- **Description**: Overcast
- **Humidity**: 30%
- **Wind Speed**: 6.39 km/h
- **Precipitation**: 0.0 mm

If you need more information, feel free to ask!

👕 Clothing Recommendations:
The current weather in New York is overcast, with a temperature of 26°C (feels like 26°C). The humidity is at 30%, and there is no precipitation. 

### Suggested Outfit:
- **Lightweight clothing**: A short-sleeve shirt or a light blouse would be comfortable in this warm temperature.
- **Layering option**: Consider a light jacket or cardigan for the overcast skies, which can make it feel a bit cooler, especially in the shade.
- **Bottoms**: Shorts or lightweight pants would work well.
- **Footwear**: Comfortable shoes, such as sneakers or sandals, are recommended since you may be out and about.
- **Accessories**: Sunglasses and a hat could be useful to protect against the sun, even 

In [9]:
# Test with London
result_london = await get_weather_and_clothing_recommendations("London")

print(f"\n🌆 {result_london['location']} Weather Information:")
print(f"{'=' * 50}")
print(result_london['weather_info'])
print(f"\n👕 Clothing Recommendations:")
print(f"{'=' * 50}")
print(result_london['clothing_recommendations'])

2025-03-29 22:41:12,577 - 8368244800 - task_runner.py-task_runner:288 - ERROR:   ❌ CrewAI config not found task.crewai_config_id is None



📖 Reading Task fef71737-6863-4d8f-a8a6-97dcb409d76c:
  Type: <class 'src.server.models.task.Task'>
  Is instance of Task: True

📖 Reading TasksManager f25216ba-b56b-49b5-84a0-a4280f5f8e83:
  Type: <class 'src.server.models.tasks_manager.TasksManager'>
  Is instance of TasksManager: True

🔍 DEBUG: get_contexts called with query='What is the weather in London?', task_id=fef71737-6863-4d8f-a8a6-97dcb409d76c

📖 Reading Task fef71737-6863-4d8f-a8a6-97dcb409d76c:
  Type: <class 'src.server.models.task.Task'>
  Is instance of Task: True
🔄 DEBUG: Syncing task contexts for task fef71737-6863-4d8f-a8a6-97dcb409d76c

🔄 DEBUG: sync_task_contexts called for task_id=fef71737-6863-4d8f-a8a6-97dcb409d76c

🔄 Syncing task contexts for fef71737-6863-4d8f-a8a6-97dcb409d76c
  Using existing task context
  Checking task attributes:
    has use_context: True
    has inherit_manager_context: True
    use_context: True
    inherit_manager_context: True
  Checking task contexts:
    has contexts attribute: Tru

✅ DEBUG: Generated 1 OpenAI embeddings
🔢 DEBUG: First embedding dimensions: 1536
🔢 DEBUG: First 5 values of first embedding: [-0.02137969806790352, -0.013100304640829563, -0.013676718808710575, 0.004907374270260334, -0.020970968529582024]
🔢 DEBUG: Generated embedding with dimensions: 1536
🔢 DEBUG: First 5 embedding values: [-0.02137969806790352, -0.013100304640829563, -0.013676718808710575, 0.004907374270260334, -0.020970968529582024]
🔎 DEBUG: Searching in default collection
🔍 DEBUG: search_in_index called with index_name=default, limit=506, filters=None
🔢 DEBUG: Embedding dimensions: 1536
🔢 DEBUG: First 5 embedding values: [-0.02137969806790352, -0.013100304640829563, -0.013676718808710575, 0.004907374270260334, -0.020970968529582024]
🔍 DEBUG: _ensure_collection called for collection_name=default
🔍 DEBUG: ChromaDB client: <chromadb.api.client.Client object at 0x15ae42c50>
📋 DEBUG: All ChromaDB collections: ['default']
✅ DEBUG: Collection default already exists in indices
📊 DEBUG: Coll

2025-03-29 22:41:15,791 - 8368244800 - task_runner.py-task_runner:288 - ERROR:   ❌ CrewAI config not found task.crewai_config_id is None



📖 Reading Task fef71737-6863-4d8f-a8a6-97dcb409d76c:
  Type: <class 'src.server.models.task.Task'>
  Is instance of Task: True

📖 Reading TasksManager f25216ba-b56b-49b5-84a0-a4280f5f8e83:
  Type: <class 'src.server.models.tasks_manager.TasksManager'>
  Is instance of TasksManager: True

🔍 DEBUG: get_contexts called with query='What should I wear for this weather in London?', task_id=fef71737-6863-4d8f-a8a6-97dcb409d76c

📖 Reading Task fef71737-6863-4d8f-a8a6-97dcb409d76c:
  Type: <class 'src.server.models.task.Task'>
  Is instance of Task: True
🔄 DEBUG: Syncing task contexts for task fef71737-6863-4d8f-a8a6-97dcb409d76c

🔄 DEBUG: sync_task_contexts called for task_id=fef71737-6863-4d8f-a8a6-97dcb409d76c

🔄 Syncing task contexts for fef71737-6863-4d8f-a8a6-97dcb409d76c
  Using existing task context
  Checking task attributes:
    has use_context: True
    has inherit_manager_context: True
    use_context: True
    inherit_manager_context: True
  Checking task contexts:
    has context

✅ DEBUG: Generated 1 OpenAI embeddings
🔢 DEBUG: First embedding dimensions: 1536
🔢 DEBUG: First 5 values of first embedding: [0.03272521495819092, 0.01430867612361908, -0.025886425748467445, -0.03779693692922592, -0.01692485809326172]
🔢 DEBUG: Generated embedding with dimensions: 1536
🔢 DEBUG: First 5 embedding values: [0.03272521495819092, 0.01430867612361908, -0.025886425748467445, -0.03779693692922592, -0.01692485809326172]
🔎 DEBUG: Searching in default collection
🔍 DEBUG: search_in_index called with index_name=default, limit=506, filters=None
🔢 DEBUG: Embedding dimensions: 1536
🔢 DEBUG: First 5 embedding values: [0.03272521495819092, 0.01430867612361908, -0.025886425748467445, -0.03779693692922592, -0.01692485809326172]
🔍 DEBUG: _ensure_collection called for collection_name=default
🔍 DEBUG: ChromaDB client: <chromadb.api.client.Client object at 0x15ae42c50>
📋 DEBUG: All ChromaDB collections: ['default']
✅ DEBUG: Collection default already exists in indices
📊 DEBUG: Collection defau


🌆 London Weather Information:
The current weather in London, United Kingdom is as follows:

- **Temperature:** 12.0°C
- **Feels Like:** 11.0°C
- **Description:** Sunny
- **Humidity:** 47%
- **Wind Speed:** 4.72 m/s
- **Precipitation:** 0.0 mm

If you need more information, feel free to ask!

👕 Clothing Recommendations:
The current weather in London is sunny with a temperature of 12°C (feels like 11°C). Here are some outfit suggestions:

- **Light Layering:** A long-sleeve shirt or a light sweater would be comfortable.
- **Jacket:** Consider a light jacket or cardigan for added warmth, especially if you're going out in the evening when it might get cooler.
- **Bottoms:** Jeans or chinos would be appropriate. If you're looking for something lighter, you could also wear breathable trousers or shorts.
- **Footwear:** Comfortable sneakers or closed shoes are a good choice.
- **Accessories:** Sunglasses might be a good idea due to the sunny weather.

Feel free to mix and match based on your